In [ ]:
from lxml import objectify, etree
import pandas as pd
from dataclasses import dataclass

aircraftIdentifications = []
NS = {
    "nas": "http://www.faa.aero/nas/3.0",
    "base": "http://www.fixm.aero/base/3.0",
    "flight": "http://www.fixm.aero/flight/3.0",
    "foundation": "http://www.fixm.aero/foundation/3.0",
    "xsi": "http://www.w3.org/2001/XMLSchema-instance",
}

def safe_get(obj, attr_path, key=None, default=None):
    """
    Safely traverse nested attributes and call .get(key) at the end if key is provided.
    attr_path: list of attribute names to traverse.
    key: key to pass to .get() at the final attribute.
    default: value to return if any attribute is missing.
    """
    try:
        for attr in attr_path:
            obj = getattr(obj, attr, None)
            if obj is None:
                return default
        if key is not None and hasattr(obj, "get"):
            return obj.get(key)
        return obj
    except Exception:
        return default

def parse_message(doc):
    
    root = objectify.fromstring(doc.encode("utf-8")) # TODO: NEED TO DEBUG THIS

    # You still typically access by navigation/XPath:
    flight = root.xpath(
        ".//*[@xsi:type and contains(@xsi:type, 'NasFlightType')]",
        namespaces=NS
    )[0]
    timestamp = safe_get(flight, [], "timestamp")
    aircraftIdentification = safe_get(flight, ["flightIdentification"], "aircraftIdentification")
    flightStatus = safe_get(flight, ["flightStatus"], "fdpsFlightStatus")
    departure = safe_get(flight, ["departure"])
    departurePoint = safe_get(departure, [], "departurePoint") if departure is not None else None
    depature_time = safe_get(departure, ["runwayPositionAndTime", "runwayTime", "actual"], "time")
    estimated_depature_time = safe_get(departure, ["runwayPositionAndTime", "runwayTime", "estimated"], "time")

    arrival = safe_get(flight, ["arrival"])
    arrivalPoint = safe_get(arrival, [], "arrivalPoint") if arrival is not None else None
    arrival_time = safe_get(arrival, ["runwayPositionAndTime", "runwayTime", "actual"], "time")
    estimated_arrival_time = safe_get(arrival, ["runwayPositionAndTime", "runwayTime", "estimated"], "time")

    # arrival_time = arrival.runwayPositionAndTime.runwayTime.actual.get("time")
    return (timestamp, aircraftIdentification, flightStatus, departurePoint, depature_time, estimated_depature_time, arrivalPoint, arrival_time, estimated_arrival_time)

def split_xml_documents(text: str):
    docs = []
    current = []

    for line in text.splitlines():
        if line.startswith("<?xml") and current:
            docs.append("\n".join(current))
            current = []
        current.append(line)

    if current:
        docs.append("\n".join(current))

    return docs

sfdps_rows = []
def process_messages_file(file_path: str):
    with open(file_path, "r", encoding="utf-8") as f:
        raw = f.read()
        docs = split_xml_documents(raw)
        for doc in docs:
            sfdps_row = parse_message(doc)
            sfdps_rows.append(sfdps_row)

In [ ]:
from pathlib import Path

dir_path = Path("../../output/swim_updating/sfdps_2026_01_21")
count = 0
for path in dir_path.iterdir():
    count+=1
    if path.is_file():
        process_messages_file(str(path))
    if count > 10:
        break
df = pd.DataFrame(
    sfdps_rows,
    columns=[
        "timestamp",
        "aircraftIdentification",
        "flightStatus",
        "departurePoint",
        "depature_time",
        "estimated_depature_time",
        "arrivalPoint",
        "arrival_time",
        "estimated_arrival_time"
    ]
)
df


In [ ]:
df[df['aircraftIdentification'] == "XAA8533"]

# Download day of s3 sfdps log data

In [ ]:
import os
import boto3
import gzip
import shutil

# ---- configuration ----
BUCKET = "planequery-swim-logs-prod"
PREFIX = "sfdps-logs/year=2026/month=01/day=21/"
DOWNLOAD_DIR = "/Users/jonahgoode/Documents/PlaneQuery/Code/planequery-data/output/swim_updating"
# -----------------------

os.makedirs(DOWNLOAD_DIR, exist_ok=True)

s3 = boto3.client("s3")

paginator = s3.get_paginator("list_objects_v2")

for page in paginator.paginate(Bucket=BUCKET, Prefix=PREFIX):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        if not key.endswith(".gz") and not key.endswith(".gzip"):
            continue

        filename = os.path.basename(key)
        local_gz_path = os.path.join(DOWNLOAD_DIR, filename)
        local_txt_path = local_gz_path.rsplit(".", 1)[0]

        # download
        s3.download_file(BUCKET, key, local_gz_path)

        # decompress
        with gzip.open(local_gz_path, "rb") as f_in:
            with open(local_txt_path, "wb") as f_out:
                shutil.copyfileobj(f_in, f_out)

        # optionally remove the .gz file
        os.remove(local_gz_path)

        print(f"Downloaded and decompressed: {local_txt_path}")
